# Classification Classique ML — Notebook 05

## Objectif
Entraîner et évaluer des classifieurs ML classiques (Random Forest, XGBoost, Logistic Regression, SVM) pour discriminer les vrais positifs (VP) des faux positifs (FP) à partir des features radiomiques.

## Prérequis
- **Librairies** : `numpy`, `pandas`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn`
- **Données** : `data/processed/ml_dataset_balanced.csv` (produit par le notebook 04)
- **Modules** : `src.O4_classical_classifier.*`

## Étapes principales
1. [Imports & configuration](#1-imports--configuration)
2. [Vérification des prérequis](#2-v%C3%A9rification-des-pr%C3%A9requis)
3. [Sélection des features](#3-s%C3%A9lection-des-features)
4. [Entraînement des classifieurs](#4-entra%C3%AEnement-des-classifieurs)
5. [Évaluation & visualisation](#5-%C3%A9valuation--visualisation)

> **Lien avec le notebook précédent :** Utilise `ml_dataset_balanced.csv` produit par `04_radiomics_demo.ipynb`.
> **Métrique principale :** **AUC-PR** (Precision-Recall) — privilégiée sur AUC-ROC pour des données fortement déséquilibrées (~5–10% de VP).


## 1. Imports & configuration

### 1.1 Imports des librairies et des modules de classification

Import de scikit-learn, XGBoost et des modules internes (`O4_classical_classifier`).
Configuration des chemins vers les données et les résultats.

> **Sortie :** variables `DATA_DIR` et `RESULTS_DIR` disponibles pour toutes les cellules.


In [ ]:
# ==============================================================
# CELLULE 1 - Imports
# ==============================================================
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = next(
    p for p in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents)
    if (p / 'src').exists() and (p / 'data').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.O3_radiomics.feature_selector import RadiomicsFeatureSelector
from src.O4_classical_classifier.train_classifier import ClassicalClassifierTrainer

DATA_DIR      = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR   = PROJECT_ROOT / 'results' / 'classical_classifier'
FIGURES_DIR   = PROJECT_ROOT / 'figures' / 'classical_classifier'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Root : {PROJECT_ROOT}')
print('Imports OK')


## 2. Vérification des prérequis

### 2.1 Contrôle de la présence du dataset ML

Vérification que `ml_dataset_balanced.csv` existe et chargement du DataFrame.
Affichage des informations de base : nombre de candidats, distribution VP/FP, nombre de features.

> **Point d'attention :** si le fichier est absent, relancer le notebook 04 cellule 4.


In [ ]:
# ==============================================================
# CELLULE 2 - Verification prerequisites
# ==============================================================
CSV_PATH = DATA_DIR / 'ml_dataset_balanced.csv'
assert CSV_PATH.exists(), f"Dataset introuvable : {CSV_PATH} -- Executer cellule 4 du notebook 04_radiomics_demo.ipynb d'abord"

df = pd.read_csv(CSV_PATH)
meta_cols = ['PatientID', 'candidate_id', 'has_embolism']
feature_cols = [c for c in df.columns if c not in meta_cols]

n_vp = int(df['has_embolism'].sum())
n_fp = len(df) - n_vp
print(f'Dataset : {len(df)} candidats | VP={n_vp} ({100*n_vp/len(df):.1f}%) | FP={n_fp} ({100*n_fp/len(df):.1f}%)')
print(f'Features brutes : {len(feature_cols)}')
print(f'Patients        : {df["PatientID"].nunique()}')
assert n_vp > 0, 'Aucun VP dans le dataset - verifier build_ml_dataset.py'
print('OK : dataset valide')

## 3. Sélection des features

### 3.1 Séparation features / cible et découpage train/test

Séparation des colonnes features (`X`) et de la variable cible (`y = has_embolism`).
Découpage stratifié train/test pour préserver la proportion VP/FP dans les deux ensembles.

> **Choix méthodologique :** le découpage stratifié est indispensable avec des données déséquilibrées pour éviter que le jeu de test ne contienne trop peu de VP.


In [ ]:
# ==============================================================
# CELLULE 3 - Selection de features
# ==============================================================
X = df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
y = df['has_embolism']

selector = RadiomicsFeatureSelector(
    variance_threshold=0.01,
    correlation_threshold=0.95,
    top_k_features=20,
    selection_method='random_forest'
)
X_selected = selector.select(X, y)
final_features = list(X_selected.columns)
print(f'Features : {len(feature_cols)} -> {len(final_features)} apres selection')
print(f'Retenues : {final_features}')

# Sauvegarder le CSV selectionne pour O4 (le trainer attend un fichier)
selected_csv = DATA_DIR / 'radiomics_features_selected.csv'
df_sel = df[meta_cols + final_features]
df_sel.to_csv(selected_csv, index=False)
print(f'CSV selectionne : {selected_csv}')


## 4. Entraînement des classifieurs

### 4.1 Entraînement de RF, XGBoost, LR et SVM

Entraînement et validation croisée de 4 classifieurs avec gestion du déséquilibre des classes :
- **Random Forest** (`class_weight='balanced'`)
- **XGBoost** (`scale_pos_weight` ajusté)
- **Logistic Regression** (`class_weight='balanced'`)
- **SVM** (`class_weight='balanced'`)

Les modèles entraînés sont sauvegardés dans `results/classical_classifier/models/`.

> **Durée estimée :** 1–5 minutes selon les hyperparamètres et la taille du dataset.


In [ ]:
# ==============================================================
# CELLULE 4 - Entrainement des classifieurs (RF, XGB, LR, SVM)
# ==============================================================
trainer = ClassicalClassifierTrainer(
    features_path=str(selected_csv),
    label_column='has_embolism',
    train_ratio=0.7,
    n_splits=5,
    random_state=42
)

models_to_test = ['random_forest', 'xgboost', 'logistic_regression', 'svm']
print(f'Modeles a tester : {models_to_test}')
print('Lancement de la validation croisee (5 folds)...')
results = trainer.train_all(models_to_test)

# Sauvegarder les modeles et resultats
trainer.save_models(str(RESULTS_DIR / 'models'))
trainer.save_results(str(RESULTS_DIR / 'results.csv'))
print('Modeles sauvegardes.')


## 5. Évaluation & visualisation

### 5.1 Comparaison des performances et courbes PR

Calcul des métriques pour chaque classifieur : AUC-PR, AUC-ROC, F1, précision, rappel.
Affichage des courbes Precision-Recall et ROC pour une comparaison visuelle.

> **Métrique clé :** **AUC-PR** — sur des données à ~5–10% de VP, l'AUC-ROC est trompeuse (un classifieur trivial peut atteindre 0.9).
> **Sortie :** `results/classical_classifier/results.csv` + figures dans `figures/`.


In [ ]:
# ==============================================================
# CELLULE 5 - Evaluation & Visualisation
# Note : privilegier AUC-PR sur donnees desequilibrees (5-10% VP)
# ==============================================================
df_res = pd.DataFrame(results).T
metrics = ['auc_roc', 'auc_pr', 'f1', 'recall', 'precision', 'accuracy']
metrics_available = [m for m in metrics if m in df_res.columns]
print('=== Resultats CV (moyenne sur 5 folds) ===')
print(df_res[metrics_available].round(3).to_string())

best_name, _ = trainer.get_best_model()
print(f'Meilleur modele (AUC-ROC) : {best_name}')

# --- AUC-ROC comparatif ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=df_res.index, y='auc_roc', data=df_res, palette='viridis', ax=axes[0])
axes[0].set_title('AUC-ROC par modele'); axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='baseline')
axes[0].legend(); axes[0].set_ylabel('AUC-ROC')

if 'auc_pr' in df_res.columns:
    sns.barplot(x=df_res.index, y='auc_pr', data=df_res, palette='magma', ax=axes[1])
    axes[1].set_title('AUC-PR par modele (reference : donnees desequilibrees)')
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('AUC-PR')
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'auc_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- Feature importance (Random Forest) ---
if 'random_forest' in trainer.models:
    rf  = trainer.models['random_forest']
    imp = pd.Series(rf.feature_importances_, index=final_features).nlargest(15)
    plt.figure(figsize=(9, 5))
    imp.sort_values().plot(kind='barh', color='teal')
    plt.title('Top 15 Features - Random Forest')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'rf_feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()


### Notes

- **AUC-PR > AUC-ROC** sur donnees desequilibrees (~5-10% VP) — privilegier AUC-PR pour comparer les modeles
- Modeles sauvegardes dans `results/classical_classifier/models/`  
- Resultats CSV dans `results/classical_classifier/results.csv`  
- Prochaine etape : notebook `06_deep_learning_demo.ipynb`